In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
refined_df = pd.read_csv("../data/processed/heart_cleaned.csv")

In [ ]:
categorical_columns = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]

In [ ]:
df_encode = pd.get_dummies(refined_df, drop_first=True, columns=categorical_columns, dtype=int)

In [ ]:
x = df_encode.drop("HeartDisease", axis=1)
y = df_encode["HeartDisease"]

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

model.fit(x_train, y_train)

In [ ]:
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report)

y_pred = model.predict(x_test)

cm = confusion_matrix(y_test, y_pred)
print(cm)
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Disease", "Disease"]
).plot(cmap="Blues")

plt.show()

In [ ]:
training_score = model.score(x_train, y_train) * 100
testing_score = model.score(x_test, y_test) * 100

print(f"Training Score: {training_score: .2f}%")
print(f"Testing Score: {testing_score: .2f}%")

In [ ]:
print(f"classification_report: {classification_report(y_test, y_pred, target_names=["No Disease", "Disease"])}")

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

y_prob_rf = model.predict_proba(x_test)[:, 1]

auc_rf = roc_auc_score(y_test, y_prob_rf)

print("Random Forest ROC-AUC:", auc_rf)

fpr, tpr, thresholds = roc_curve(y_test, y_prob_rf)

plt.plot(fpr, tpr, label=f"AUC = {auc_rf:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Random Forest")
plt.legend()
plt.show()

# Testing with same previous decision tree training and testing dataset

In [ ]:
import joblib
new_x_train = joblib.load("X_train.pkl")
new_x_test = joblib.load("X_test.pkl")
new_y_train = joblib.load("y_train.pkl")
new_y_test = joblib.load("y_test.pkl")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(new_x_train, new_y_train)

In [ ]:
new_y_pred = rf_model.predict(new_x_test)
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_auc_score, roc_curve, RocCurveDisplay)

new_training_score = rf_model.score(new_x_train, new_y_train) * 100
new_testing_score = rf_model.score(new_x_test, new_y_test) * 100

print(f"New training score: {new_training_score: .2f}%")
print(f"New testing score: {new_testing_score: .2f}%")

In [ ]:
joblib.dump(rf_model, "random_forest.pkl")